In [4]:
import pandas as pd
import sqlite3

# Load dataset
df = pd.read_csv('Sample - Superstore.csv', encoding='ISO-8859-1')

# Fix column headers for SQL compatibility
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

# Connect to SQLite memory engine
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Import raw data
df.to_sql('superstore_raw', conn, index=False, if_exists='replace')

# Create tables
cursor.execute("DROP TABLE IF EXISTS customers;")
cursor.execute("DROP TABLE IF EXISTS products;")
cursor.execute("DROP TABLE IF EXISTS orders;")

cursor.execute("""
CREATE TABLE customers (
    Customer_ID VARCHAR(50) PRIMARY KEY,
    Customer_Name VARCHAR(100),
    Segment VARCHAR(50)
);
""")

cursor.execute("""
CREATE TABLE products (
    Product_ID VARCHAR(50) PRIMARY KEY,
    Product_Name VARCHAR(255),
    Category VARCHAR(50),
    Sub_Category VARCHAR(50)
);
""")

cursor.execute("""
CREATE TABLE orders (
    Row_ID INT PRIMARY KEY,
    Order_ID VARCHAR(50),
    Order_Date VARCHAR(50),
    Ship_Date VARCHAR(50),
    Ship_Mode VARCHAR(50),
    Customer_ID VARCHAR(50),
    Region VARCHAR(50),
    Product_ID VARCHAR(50),
    Sales DECIMAL(10,2),
    Quantity INT,
    Discount DECIMAL(4,2),
    Profit DECIMAL(10,2)
);
""")

# Insert data using SELECT DISTINCT
cursor.execute("""
INSERT OR IGNORE INTO customers (Customer_ID, Customer_Name, Segment)
SELECT DISTINCT Customer_ID, Customer_Name, Segment FROM superstore_raw;
""")

cursor.execute("""
INSERT OR IGNORE INTO products (Product_ID, Product_Name, Category, Sub_Category)
SELECT Product_ID, Product_Name, Category, Sub_Category FROM superstore_raw GROUP BY Product_ID;
""")

cursor.execute("""
INSERT OR IGNORE INTO orders
SELECT Row_ID, Order_ID, Order_Date, Ship_Date, Ship_Mode, Customer_ID, Region, Product_ID, Sales, Quantity, Discount, Profit
FROM superstore_raw;
""")

conn.commit()

# Output utility function
def run_query(title, sql):
    print(f"\n>>> {title}")
    display(pd.read_sql_query(sql, conn))

print("Tables loaded successfully.")

Tables loaded successfully.


In [5]:
# --- STEP 2: PERFORM REQUIRED QUERIES ---

# 1. Orders with sales greater than average (Subquery)
run_query("1. Orders with Sales > Average Sales", """
SELECT Order_ID, Customer_ID, Sales
FROM orders
WHERE Sales > (SELECT AVG(Sales) FROM orders)
ORDER BY Sales DESC LIMIT 5;
""")

# 2. Highest sales order for each customer (Subquery)
run_query("2. Highest Sales Order Per Customer", """
SELECT Order_ID, Customer_ID, Sales
FROM orders o
WHERE Sales = (SELECT MAX(Sales) FROM orders sub WHERE sub.Customer_ID = o.Customer_ID)
ORDER BY Sales DESC LIMIT 5;
""")

# 3. Calculate total sales for each customer (CTE)
run_query("3. Total Sales Per Customer", """
WITH CustomerSales AS (
    SELECT Customer_ID, ROUND(SUM(Sales), 2) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT * FROM CustomerSales LIMIT 5;
""")

# 4. Find customers whose total sales are above average (CTE + Subquery)
run_query("4. Customers with Total Sales Above Average", """
WITH CustomerTotals AS (
    SELECT Customer_ID, SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT Customer_ID, ROUND(Total_Sales, 2) AS Total_Sales
FROM CustomerTotals
WHERE Total_Sales > (SELECT AVG(Total_Sales) FROM CustomerTotals)
ORDER BY Total_Sales DESC LIMIT 5;
""")

# 5. Rank all customers based on total sales (Window Function)
run_query("5. Rank Customers Based on Total Sales", """
WITH CustomerTotals AS (
    SELECT Customer_ID, SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT Customer_ID, ROUND(Total_Sales, 2) AS Total_Sales,
       RANK() OVER (ORDER BY Total_Sales DESC) as Customer_Rank
FROM CustomerTotals LIMIT 5;
""")

# 6. Assign row numbers to each order within a customer (Window Function + PARTITION BY)
run_query("6. Row Numbers Per Order Within Customer", """
SELECT Customer_ID, Order_ID, Sales,
       ROW_NUMBER() OVER (PARTITION BY Customer_ID ORDER BY Sales DESC) as Order_Seq
FROM orders LIMIT 10;
""")

# 7. Display top 3 customers based on total sales (Window Function)
run_query("7. Top 3 Customers via Window Ranking", """
WITH RankedCustomers AS (
    SELECT Customer_ID, SUM(Sales) AS Total_Sales,
           RANK() OVER (ORDER BY SUM(Sales) DESC) as Customer_Rank
    FROM orders
    GROUP BY Customer_ID
)
SELECT * FROM RankedCustomers WHERE Customer_Rank <= 3;
""")


# --- STEP 3: FINAL COMBINED QUERY (JOIN + CTE + Window Function) ---
run_query("Step 3: Final Combined Analysis Matrix", """
WITH CustomerTotals AS (
    SELECT Customer_ID, SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
),
RankedCustomers AS (
    SELECT Customer_ID, Total_Sales,
           DENSE_RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Rank
    FROM CustomerTotals
)
SELECT rc.Sales_Rank, c.Customer_Name, ROUND(rc.Total_Sales, 2) AS Total_Sales
FROM RankedCustomers rc
JOIN customers c ON rc.Customer_ID = c.Customer_ID
WHERE rc.Sales_Rank <= 10
ORDER BY rc.Sales_Rank ASC;
""")


# --- MINI PROJECT: CUSTOMER SALES INSIGHTS ---

# 1. Who are the top 5 customers?
run_query("Mini Project 1: Top 5 Customers", """
SELECT c.Customer_Name, ROUND(SUM(o.Sales), 2) AS Total_Sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY o.Customer_ID
ORDER BY Total_Sales DESC LIMIT 5;
""")

# 2. Who are the bottom 5 customers?
run_query("Mini Project 2: Bottom 5 Customers", """
SELECT c.Customer_Name, ROUND(SUM(o.Sales), 2) AS Total_Sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY o.Customer_ID
ORDER BY Total_Sales ASC LIMIT 5;
""")

# 3. Which customers made only one order?
run_query("Mini Project 3: Single-Order Customers", """
SELECT c.Customer_Name, COUNT(DISTINCT o.Order_ID) AS Order_Count
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY o.Customer_ID
HAVING Order_Count = 1 LIMIT 5;
""")

# 4. Which customers have above-average sales?
run_query("Mini Project 4: Customers with Above-Average Sales", """
WITH CustomerTotals AS (
    SELECT Customer_ID, SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT Customer_ID, ROUND(Total_Sales, 2) AS Total_Sales
FROM CustomerTotals
WHERE Total_Sales > (SELECT AVG(Total_Sales) FROM CustomerTotals)
LIMIT 5;
""")

# 5. What is the highest order value per customer?
run_query("Mini Project 5: Highest Order Value Per Customer", """
SELECT Customer_ID, ROUND(MAX(Sales), 2) AS Max_Order_Value
FROM orders
GROUP BY Customer_ID
LIMIT 5;
""")


>>> 1. Orders with Sales > Average Sales


,Order_ID,Customer_ID,Sales
0,CA-2014-145317,SM-20320,22638.480
1,CA-2016-118689,TC-20980,17499.950
2,CA-2017-140151,RB-19360,13999.960
3,CA-2017-127180,TA-21385,11199.968
4,CA-2017-166709,HL-15040,10499.970



>>> 2. Highest Sales Order Per Customer


,Order_ID,Customer_ID,Sales
0,CA-2014-145317,SM-20320,22638.480
1,CA-2016-118689,TC-20980,17499.950
2,CA-2017-140151,RB-19360,13999.960
3,CA-2017-127180,TA-21385,11199.968
4,CA-2017-166709,HL-15040,10499.970



>>> 3. Total Sales Per Customer


...

,Customer_Name,Order_Count
0,Anthony O'Donnell,1
1,Anemone Ratner,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1



>>> Mini Project 4: Customers with Above-Average Sales


,Customer_ID,Total_Sales
0,AA-10315,5563.56
1,AA-10645,5086.94
2,AB-10060,7755.62
3,AB-10105,14473.57
4,AC-10450,5527.85



>>> Mini Project 5: Highest Order Value Per Customer


,Customer_ID,Max_Order_Value
0,AA-10315,3930.07
1,AA-10375,499.98
2,AA-10480,479.97
3,AA-10645,1323.90
4,AB-10015,341.96
